# Funciones Recursivas

**Curso:** Programación Avanzada  
**Kernel:** xeus-cling (C++17)

---

En este cuaderno vamos a explorar uno de los conceptos más elegantes de la programación:

- **Recursión**: una función que se llama a sí misma para resolver un problema
- **Caso base**: la condición que detiene las llamadas
- **Caso recursivo**: el paso que acerca el problema al caso base
- **Call stack**: cómo la memoria administra las llamadas encadenadas

La idea central es simple: **si sabes resolver un problema pequeño, y sabes cómo reducir el problema grande al pequeño, ya tienes una solución recursiva**.

---

## Requisitos previos

Antes de comenzar, ejecuta esta celda para cargar las librerías:

In [ ]:
#include <iostream>
using namespace std;

---

## Parte 1 — ¿Qué es una función recursiva?

### 1.1 La idea

Imagina que te piden contar cuántos escalones hay en una escalera muy larga.  
Una estrategia: pararte en el primer escalón y decir:

> "Este escalón cuenta como 1, más los que hay después de mí."

Y la persona en el siguiente escalón dice lo mismo, y la siguiente, y así hasta que alguien llega al último escalón y dice:

> "Yo soy el último — hay 0 escalones después de mí."

Eso es **recursión**: resolver el problema desde mi posición asumiendo que alguien más ya resolvió el resto.

En código, una función recursiva se ve así:

```
función(problema) {
    si el problema es trivial → devuelve la respuesta directa   ← caso base
    si no               → hace algo + llama a función(problema más pequeño)   ← caso recursivo
}
```

### 1.2 Primer ejemplo: cuenta regresiva

In [ ]:
void cuenta(int n) {
    if (n == 0) {           // caso base: ya llegamos a cero
        cout << "¡Despegue!" << endl;
        return;
    }
    cout << n << "..." << endl;
    cuenta(n - 1);          // caso recursivo: resolver para n-1
}

cuenta(5);

5...
4...
3...
2...
1...
¡Despegue!


**¿Qué pasó?**  
Cada llamada imprime su número y luego llama a `cuenta` con el número anterior.  
Cuando `n` llega a 0, la función no se llama más: retorna sin hacer nada recursivo.

Sin el caso base (`if (n == 0)`), la función se llamaría infinitamente — lo veremos más adelante.

---

## Parte 2 — Las dos partes obligatorias

Toda función recursiva correcta tiene **exactamente** estas dos partes:

| Parte | ¿Qué hace? | ¿Qué pasa si falta? |
|---|---|---|
| **Caso base** | Detiene la recursión y devuelve un resultado directo | La función se llama infinitamente → **stack overflow** |
| **Caso recursivo** | Llama a la función con un argumento *más cercano* al caso base | La función no hace nada útil |

### 2.1 Suma acumulada: `suma(n) = 1 + 2 + ... + n`

La fórmula recursiva es:

```
suma(0) = 0                    ← caso base
suma(n) = n + suma(n - 1)      ← caso recursivo
```

Esto se lee: "la suma del 1 al n es n más la suma del 1 al n-1".

In [ ]:
int suma(int x) {
    if (x == 0)          // caso base
        return 0;
    return x + suma(x - 1);   // caso recursivo
}

cout << "suma(10) = " << suma(10) << endl;
cout << "suma(5)  = " << suma(5)  << endl;
cout << "suma(1)  = " << suma(1)  << endl;

suma(10) = 55
suma(5)  = 15
suma(1)  = 1


**Traza de `suma(4)` paso a paso:**

```
suma(4)
  └─ 4 + suma(3)
           └─ 3 + suma(2)
                    └─ 2 + suma(1)
                             └─ 1 + suma(0)
                                      └─ 0    ← caso base, regresa 0

Regresando:
  suma(0) = 0
  suma(1) = 1 + 0 = 1
  suma(2) = 2 + 1 = 3
  suma(3) = 3 + 3 = 6
  suma(4) = 4 + 6 = 10  ✓
```

---

## Parte 3 — El call stack: la memoria detrás de la recursión

### 3.1 ¿Qué es el call stack?

Cada vez que una función se llama, el sistema reserva un bloque de memoria llamado **stack frame** (o marco de pila).  
Ese bloque guarda:
- Los parámetros de la función (`x = 4`, `x = 3`, etc.)
- Las variables locales
- La dirección de retorno (a dónde volver cuando la función termine)

Los frames se **apilan** uno sobre otro (como platos) y se **desapilan** en orden inverso conforme las funciones retornan.

### 3.2 La pila durante `suma(3)`

```
  Estado JUSTO ANTES de que suma(0) retorne:

  ┌─────────────────────────┐  ← tope (el más reciente)
  │  suma(0)   x=0          │  retorna 0 inmediatamente
  ├─────────────────────────┤
  │  suma(1)   x=1          │  espera el resultado de suma(0)
  ├─────────────────────────┤
  │  suma(2)   x=2          │  espera el resultado de suma(1)
  ├─────────────────────────┤
  │  suma(3)   x=3          │  espera el resultado de suma(2)
  ├─────────────────────────┤
  │  main()                 │  espera el resultado de suma(3)
  └─────────────────────────┘  ← fondo

  Después, los frames se desapilan:
  suma(0)→0  suma(1)→1  suma(2)→3  suma(3)→6  main imprime 6
```

### 3.3 Verificar la profundidad: versión con traza

In [ ]:
// Versión que imprime cada llamada para ver el stack en acción
int sumaTraza(int x, int nivel = 0) {
    // Indentación proporcional al nivel de profundidad
    for (int i = 0; i < nivel; i++) cout << "  ";
    cout << "▶ suma(" << x << ") llamada" << endl;

    if (x == 0) {
        for (int i = 0; i < nivel; i++) cout << "  ";
        cout << "◀ suma(0) retorna 0  [caso base]" << endl;
        return 0;
    }

    int resultado = x + sumaTraza(x - 1, nivel + 1);

    for (int i = 0; i < nivel; i++) cout << "  ";
    cout << "◀ suma(" << x << ") retorna " << resultado << endl;
    return resultado;
}

sumaTraza(4);

▶ suma(4) llamada
  ▶ suma(3) llamada
    ▶ suma(2) llamada
      ▶ suma(1) llamada
        ▶ suma(0) llamada
        ◀ suma(0) retorna 0  [caso base]
      ◀ suma(1) retorna 1
    ◀ suma(2) retorna 3
  ◀ suma(3) retorna 6
◀ suma(4) retorna 10


La indentación muestra exactamente el call stack: cuanto más profundo a la izquierda, más arriba en la pila.  
El árbol de llamadas tiene la misma forma que la pila de frames.

---

## Parte 4 — Sin caso base: stack overflow

¿Qué pasa si quitamos el caso base de `suma`?

```cpp
int sumaSinBase(int x) {
    return x + sumaSinBase(x - 1);   // ← sin condición de parada
}
```

Cada llamada agrega un frame a la pila, que crece sin parar:

```
  sumaSinBase(10)
  sumaSinBase(9)
  sumaSinBase(8)
  ...
  sumaSinBase(-10000)
  ...          ← la pila se llena completamente
  ¡CRASH!  Stack overflow
```

> **No ejecutes la celda de abajo tal cual** — causaría que el kernel se cuelgue.  
> La función está definida pero no llamada, para que puedas leer el código sin riesgo.

In [ ]:
// ⚠ NO LLAMAR — definición solo para leer
int sumaSinBase(int x) {
    return x + sumaSinBase(x - 1);   // recursión infinita: no hay caso base
}

// Si ejecutaras: sumaSinBase(10);
// El programa crashearía con: "Segmentation fault" o "Stack overflow"
cout << "Función definida — no se llamó." << endl;

Función definida — no se llamó.


**Regla de oro:** antes de escribir la llamada recursiva, pregúntate:  
*¿El argumento que paso está más cerca del caso base que el argumento que recibí?*  
Si la respuesta es no (o no siempre), tu función puede entrar en recursión infinita.

---

## Parte 5 — Potencias

### 5.1 Potencias de 3

La fórmula recursiva:

```
pot3(0) = 1          ← caso base: 3⁰ = 1
pot3(n) = 3 * pot3(n-1)   ← caso recursivo: 3ⁿ = 3 × 3ⁿ⁻¹
```

In [ ]:
int pot3(int n) {
    if (n == 0) return 1;        // caso base
    return 3 * pot3(n - 1);      // caso recursivo
}

for (int i = 0; i <= 6; i++)
    cout << "3^" << i << " = " << pot3(i) << endl;

3^0 = 1
3^1 = 3
3^2 = 9
3^3 = 27
3^4 = 81
3^5 = 243
3^6 = 729


### 5.2 Generalizar: cualquier base

Solo cambiamos el `3` hardcodeado por un parámetro `base`:

```
pot(base, 0)  = 1
pot(base, n)  = base × pot(base, n-1)
```

In [ ]:
int pot(int base, int n) {
    if (n == 0) return 1;
    return base * pot(base, n - 1);
}

cout << "2^10 = " << pot(2, 10) << endl;
cout << "5^4  = " << pot(5, 4)  << endl;
cout << "7^0  = " << pot(7, 0)  << endl;

2^10 = 1024
5^4  = 625
7^0  = 1


---

## Parte 6 — Factorial

El factorial es el ejemplo más clásico de recursión:

```
0! = 1           ← caso base (por definición matemática)
n! = n × (n-1)!  ← caso recursivo
```

Traza de `factorial(5)`:

```
factorial(5)
  = 5 × factorial(4)
  = 5 × 4 × factorial(3)
  = 5 × 4 × 3 × factorial(2)
  = 5 × 4 × 3 × 2 × factorial(1)
  = 5 × 4 × 3 × 2 × 1 × factorial(0)
  = 5 × 4 × 3 × 2 × 1 × 1
  = 120
```

In [ ]:
long long factorial(int n) {
    if (n == 0) return 1;           // caso base
    return n * factorial(n - 1);    // caso recursivo
}

for (int i = 0; i <= 10; i++)
    cout << i << "! = " << factorial(i) << endl;

0! = 1
1! = 1
2! = 2
3! = 6
4! = 24
5! = 120
6! = 720
7! = 5040
8! = 40320
9! = 362880
10! = 3628800


**¿Por qué `long long` y no `int`?**  
Los factoriales crecen muy rápido. `13!` ya supera el rango de un `int` de 32 bits.  
Con `long long` llegamos hasta aproximadamente `20!` antes de desbordar.

---

## Parte 7 — Recursivo vs. Iterativo

Toda función recursiva puede reescribirse con un ciclo, y viceversa.  
¿Cuál conviene usar?

| Criterio | Recursivo | Iterativo |
|---|---|---|
| Legibilidad | Alta cuando el problema es naturalmente recursivo | Alta cuando hay un ciclo obvio |
| Uso de memoria | Cada llamada usa un frame del stack | Solo usa las variables del ciclo |
| Riesgo de overflow | Sí — profundidad limitada por el stack | No |
| Casos típicos | Árboles, backtracking, divide y vencerás | Sumas, búsquedas simples, arrays |

### Suma: versión iterativa para comparar

In [ ]:
// Iterativa
int sumaIterativa(int n) {
    int total = 0;
    for (int i = 1; i <= n; i++)
        total += i;
    return total;
}

// Recursiva (ya definida arriba)
// int suma(int x) { if (x==0) return 0; return x + suma(x-1); }

cout << "Iterativa suma(100) = " << sumaIterativa(100) << endl;
cout << "Recursiva suma(100) = " << suma(100)          << endl;

Iterativa suma(100) = 5050
Recursiva suma(100) = 5050


---

## Ejercicios

### Ejercicio 1 — Suma de dígitos

Escribe una función recursiva `sumaDigitos(n)` que devuelva la suma de los dígitos de un entero positivo.

Ejemplos: `sumaDigitos(1234) = 10`, `sumaDigitos(99) = 18`, `sumaDigitos(7) = 7`

*Pista: el último dígito de `n` es `n % 10`. El número sin su último dígito es `n / 10`.*

In [ ]:
// Tu solución aquí


### Ejercicio 2 — Potencia con exponente negativo

Extiende la función `pot(base, n)` para que también acepte exponentes negativos.  
Recuerda que `base^(-n) = 1 / base^n`.

Prueba con: `pot(2, -3) = 0.125`, `pot(10, -2) = 0.01`

In [ ]:
// Tu solución aquí


### Ejercicio 3 — Contar ceros

Escribe `contarCeros(n)` que cuente cuántos dígitos cero tiene el entero `n`.

Ejemplos: `contarCeros(1001) = 2`, `contarCeros(300) = 2`, `contarCeros(5) = 0`

In [ ]:
// Tu solución aquí


---

## Resumen

| Concepto | Descripción |
|---|---|
| **Función recursiva** | Función que se llama a sí misma para resolver un subproblema |
| **Caso base** | Condición que detiene la recursión; devuelve un valor directo |
| **Caso recursivo** | Llama a la función con un argumento más cercano al caso base |
| **Call stack** | Pila de frames de memoria; uno por cada llamada activa |
| **Stack frame** | Bloque que guarda parámetros, variables locales y dirección de retorno |
| **Stack overflow** | Error cuando la pila se llena; ocurre sin caso base o con recursión muy profunda |
| **Profundidad** | Número máximo de llamadas activas simultáneas |

**Patrón general de una función recursiva correcta:**

```cpp
tipo funcion(parametros) {
    if (condición de caso base)
        return valor_directo;              // ← detiene la recursión
    return operación + funcion(más_pequeño);  // ← acerca al caso base
}
```

---
*Programación Avanzada — Universidad Panamericana*